In [1]:
import pandas as pd, numpy as np, xgboost
print(pd.__version__, xgboost.__version__)

3.1.0 3.4.1


In [3]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")

url = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@localhost:5432/{os.getenv('POSTGRES_DB')}"
)
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM txn_features", engine)
print(df.shape)
df.head()

(1296675, 11)


,trans_id,trans_ts,card_id,category,amount,is_fraud,hour_of_day,secs_since_last,txn_count_1h,avg_amount_prior,amount_ratio
0,1018,2019-01-01 12:47:15,60416207185,misc_net,7.27,0,12,NaN,0,NaN,NaN
1,2725,2019-01-02 08:44:57,60416207185,gas_transport,52.94,0,8,71862.0,0,7.270,7.281981
2,2727,2019-01-02 08:47:36,60416207185,gas_transport,82.08,0,8,159.0,1,30.105,2.726457
3,2883,2019-01-02 12:38:14,60416207185,kids_pets,34.79,0,12,13838.0,0,47.430,0.733502
4,2908,2019-01-02 13:10:46,60416207185,home,27.18,0,13,1952.0,1,44.270,0.613960


In [4]:
num_cols = ["amount", "secs_since_last", "avg_amount_prior", "amount_ratio"]
df[num_cols] = df[num_cols].apply(pd.to_numeric)
df["category"] = df["category"].astype("category")

print(df.shape)
print("fraud rate:", df["is_fraud"].mean())
print(df.isna().sum())
print(df["trans_ts"].min(), "to", df["trans_ts"].max())
print(df.dtypes)

(1296675, 11)
fraud rate: 0.005788651743883394
trans_id              0
trans_ts              0
card_id               0
category              0
amount                0
is_fraud              0
hour_of_day           0
secs_since_last     983
txn_count_1h          0
avg_amount_prior    983
amount_ratio        983
dtype: int64
2019-01-01 00:00:18 to 2020-06-21 12:13:37
trans_id                     int64
trans_ts            datetime64[us]
card_id                      int64
category                  category
amount                     float64
is_fraud                     int64
hour_of_day                  int64
secs_since_last            float64
txn_count_1h                 int64
avg_amount_prior           float64
amount_ratio               float64
dtype: object


In [5]:
cutoff = "2020-03-01"
train = df[df["trans_ts"] < cutoff]
test = df[df["trans_ts"] >= cutoff]

FEATURES = [
    "amount", "hour_of_day", "secs_since_last",
    "txn_count_1h", "avg_amount_prior", "amount_ratio", "category",
]

X_train, y_train = train[FEATURES], train["is_fraud"]
X_test, y_test = test[FEATURES], test["is_fraud"]

print("train rows:", len(train), "| fraud rate:", y_train.mean())
print("test rows: ", len(test), "| fraud rate:", y_test.mean())

train rows: 1024843 | fraud rate: 0.005756003602503018
test rows:  271832 | fraud rate: 0.00591173960387298


In [6]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(ratio, 1))

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=ratio,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]
print("PR-AUC :", average_precision_score(y_test, scores))
print("ROC-AUC:", roc_auc_score(y_test, scores))

scale_pos_weight: 172.7
PR-AUC : 0.9010281558352562
ROC-AUC: 0.9984607229115979


In [7]:
from sklearn.metrics import precision_score, recall_score

for t in [0.3, 0.5, 0.8, 0.9, 0.95, 0.99]:
    pred = (scores >= t).astype(int)
    p = precision_score(y_test, pred, zero_division=0)
    r = recall_score(y_test, pred)
    print(f"threshold {t}: precision {p:.3f} | recall {r:.3f} | alerts {pred.sum()}")

print("actual frauds in test:", y_test.sum())

threshold 0.3: precision 0.321 | recall 0.969 | alerts 4857
threshold 0.5: precision 0.418 | recall 0.949 | alerts 3644
threshold 0.8: precision 0.604 | recall 0.912 | alerts 2425
threshold 0.9: precision 0.705 | recall 0.880 | alerts 2007
threshold 0.95: precision 0.795 | recall 0.841 | alerts 1700
threshold 0.99: precision 0.920 | recall 0.741 | alerts 1295
actual frauds in test: 1607
